<a href="https://colab.research.google.com/github/AnshG-19/Blackhole_simulator_rayTracing/blob/main/notebooks/06_interactive_visualization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.patches import Circle
from ipywidgets import interact, FloatSlider

Define the black hole and coordinate grid
Explanation
We use geometric units so the mass scale is \(M=1\). This means the event horizon is at \(r=2\), the photon sphere at \(r=3\), and the simplified disk inner edge at \(r=6\).

X and Y represent every pixel's coordinates. R gives the distance of each pixel from the center. We'll use that distance to determine how bright each part of the disk should be.

Unlike the earlier notebooks, this grid is created once and reused whenever you move a slider.

In [2]:
# Use geometric units: G = c = M = 1
M = 1.0

# Characteristic radii
r_s = 2 * M
r_ph = 3 * M

# Accretion disk inner edge (simplified Schwarzschild model)
r_inner = 6 * M

# Create a 2D coordinate grid
resolution = 500
view_radius = 25

x = np.linspace(-view_radius, view_radius, resolution)
y = np.linspace(-view_radius, view_radius, resolution)

X, Y = np.meshgrid(x, y)

# Distance from the black hole at every grid point
R = np.sqrt(X**2 + Y**2)

print("Event horizon radius:", r_s)
print("Photon sphere radius:", r_ph)
print("Disk inner radius:", r_inner)

Event horizon radius: 2.0
Photon sphere radius: 3.0
Disk inner radius: 6.0


Create the interactive visualization function

In [3]:
def show_black_hole(
    outer_radius=20.0,
    brightness_scale=1.0,
    asymmetry=0.3
):
    # Only pixels inside the disk boundaries belong to the disk
    disk_mask = (R >= r_inner) & (R <= outer_radius)

    # Start with a completely dark image
    brightness = np.zeros_like(R)

    # Simplified radial brightness profile
    brightness[disk_mask] = (
        (r_inner / R[disk_mask])**3
        * (1 - np.sqrt(r_inner / R[disk_mask]))
    )

    # Add an angular pattern to make the disk less uniform
    phi = np.arctan2(Y, X)

    angular_pattern = (
        1 + asymmetry * np.cos(phi)
    )

    brightness *= angular_pattern

    # Adjust brightness without modifying the original formula
    brightness *= brightness_scale

    # Display the image
    fig, ax = plt.subplots(figsize=(8, 8))

    image = ax.imshow(
        brightness,
        extent=[
            -view_radius, view_radius,
            -view_radius, view_radius
        ],
        origin="lower",
        cmap="inferno",
        vmin=0,
        vmax=max(0.01, brightness_scale)
    )

    # Draw the event horizon
    event_horizon = Circle(
        (0, 0),
        r_s,
        color="black",
        zorder=3
    )
    ax.add_patch(event_horizon)

    # Draw the photon sphere
    photon_sphere = Circle(
        (0, 0),
        r_ph,
        fill=False,
        edgecolor="cyan",
        linestyle="--",
        linewidth=1.5,
        zorder=4
    )
    ax.add_patch(photon_sphere)

    ax.set_title(
        f"Black Hole Accretion Disk\n"
        f"Outer radius = {outer_radius:.1f} M"
    )
    ax.set_xlabel("x (units of M)")
    ax.set_ylabel("y (units of M)")
    ax.set_aspect("equal")

    fig.colorbar(image, ax=ax, label="Relative brightness")
    plt.show()

Add the sliders

In [4]:
interact(
    show_black_hole,

    outer_radius=FloatSlider(
        value=20.0,
        min=10.0,
        max=24.0,
        step=1.0,
        description="Disk radius:",
        continuous_update=False
    ),

    brightness_scale=FloatSlider(
        value=1.0,
        min=0.2,
        max=2.0,
        step=0.1,
        description="Brightness:",
        continuous_update=False
    ),

    asymmetry=FloatSlider(
        value=0.3,
        min=0.0,
        max=0.8,
        step=0.1,
        description="Asymmetry:",
        continuous_update=False
    )
);

interactive(children=(FloatSlider(value=20.0, continuous_update=False, description='Disk radius:', max=24.0, m…

## Observations

1. Increasing the outer radius extends the disk outward.
2. Changing brightness affects the displayed intensity.
3. Increasing asymmetry makes one side brighter than the other.

## Limitations

This visualization is a simplified 2D brightness model. It does not
simulate light bending, photon capture, orbital motion, gravitational
redshift, or relativistic Doppler effects.